# 03 — Base TabEBM 샘플 생성 (SGLD 파라미터 sweep)

**Step 3/4**: 기본 TabEBM 의 내부 SGLD 파라미터를 조절하며 synthetic 데이터 생성.

VP-SGLD 와 달리 ensemble 을 사용하지 않고, 단일 EBM 에서 직접 SGLD 를 돌립니다.

| 파라미터 | 기본값 | 의미 |
|----------|--------|------|
| `sgld_steps` | 200 | SGLD 반복 수 |
| `sgld_step_size` | 0.1 | gradient step size η |
| `sgld_noise_std` | 0.01 | SGLD noise 크기 |
| `starting_point_noise_std` | 0.01 | 초기 perturbation σ |
| `distance_negative_class` | 5.0 | surrogate negative 거리 α |

## 0. Setup

In [20]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp
try: _mp.set_start_method('spawn', force=True)
except RuntimeError: pass

os.chdir('/home/work/JooKyung/TabEBM')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')

import numpy as np

def _fmt(s):
    s = int(s); m, s = divmod(s, 60)
    return f'{m}m{s:02d}s' if m else f'{s}s'

print('ready')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ready


## 1. eval_dir 선택 (01 에서 생성)

In [21]:
fair_root = Path('experiments/fair_eval')
if fair_root.exists():
    for p in sorted(fair_root.iterdir()):
        if p.is_dir() and (p / 'config.json').exists():
            cfg = json.loads((p / 'config.json').read_text())
            ens_ok = (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
            smp_ok = (p / 'samples').exists()
            print(f'  {p.name}  methods={cfg["methods"]} K={cfg["K"]}  '
                  f'ens={"OK" if ens_ok else "--"}  samples={"OK" if smp_ok else "--"}')
else:
    print('  (없음 — 01 먼저 실행)')

  20260417_215937  methods=['Distance'] K=10  ens=OK  samples=OK


In [ ]:
_candidates = sorted([
    p for p in fair_root.iterdir()
    if p.is_dir() and (p / 'config.json').exists()
       and (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
])
EVAL_DIR = _candidates[-1] if _candidates else Path('experiments/fair_eval/NONE')
# EVAL_DIR = Path('experiments/fair_eval/20260417_215937')   # <-- legacy stock (재현 불가)
assert (EVAL_DIR / 'config.json').exists(), f'없음: {EVAL_DIR}'

config = json.loads((EVAL_DIR / 'config.json').read_text())
print(f'EVAL_DIR: {EVAL_DIR}')
print(f'  {config["n_splits"]} splits, {config["n_samples"]} samples, '
      f'{config["n_features"]}d, classes={config["classes"]}')

## 2. TabEBM SGLD 하이퍼파라미터

각 파라미터를 별도 셀에서 조절합니다.

In [23]:
# === 2-1. sgld_steps (SGLD 반복 수) ===
# 논문 기본값: 200
SGLD_STEPS = [200]
# SGLD_STEPS = [50, 100, 200]              # sweep
# SGLD_STEPS = [50, 100, 200, 500]
print(f'SGLD_STEPS = {SGLD_STEPS}')

SGLD_STEPS = [200]


In [24]:
# === 2-2. sgld_step_size (gradient step size η) ===
# 논문 기본값: 0.1
# SGLD_STEP_SIZE = [0.1]
SGLD_STEP_SIZE = [0.1, 1]   # sweep
# SGLD_STEP_SIZE = [0.01, 0.1, 0.5]
print(f'SGLD_STEP_SIZE = {SGLD_STEP_SIZE}')

SGLD_STEP_SIZE = [0.1, 1]


In [25]:
# === 2-3. sgld_noise_std (SGLD noise 크기) ===
# 논문 기본값: 0.01
SGLD_NOISE_STD = [0.01]
# SGLD_NOISE_STD = [0.001, 0.01, 0.1]        # sweep
# SGLD_NOISE_STD = [0.001, 0.005, 0.01, 0.05]
print(f'SGLD_NOISE_STD = {SGLD_NOISE_STD}')

SGLD_NOISE_STD = [0.01]


In [26]:
# === 2-4. starting_point_noise_std (초기 perturbation) ===
# 논문 기본값: 0.01
STARTING_NOISE = [0.01]
# STARTING_NOISE = [0.001, 0.01, 0.1]         # sweep
# STARTING_NOISE = [0.001, 0.01, 0.05, 0.1]
print(f'STARTING_NOISE = {STARTING_NOISE}')

STARTING_NOISE = [0.01]


In [ ]:
# === 2-5. distance_negative_class (surrogate negative 거리 α) ===
# 논문 기본값: 5.0 (Section 2.2, Appendix B.6)
DIST_NEG = [5.0]
# DIST_NEG = [1.0, 3.0, 5.0, 10.0]            # sweep
# DIST_NEG = [2.0]                             # 탐색값
# DIST_NEG = [2.0, 5.0, 10.0]
print(f'DIST_NEG = {DIST_NEG}')


In [ ]:
# === 2-6. SEED sweep (corner randomness 다변화) ===
# 같은 split / 같은 SGLD 파라미터 에서 seed 만 바꿔 K_single 개 서로 다른 corner set 생성.
# 효과: 01 ensemble 의 per-member corner_seed (42, 43, 44, ...) 처럼
#       같은 데이터/같은 SGLD 설정인데 4개 corner 만 바꿔 가며 여러 sample set 확보.
# worker 에서 seed = split_seed + seed_offset 으로 사용됨.
SEED_OFFSETS = [0]
# SEED_OFFSETS = [0, 1, 2, 3, 4]                      # 5 회 반복
# SEED_OFFSETS = list(range(10))                       # 10 회
print(f'SEED_OFFSETS = {SEED_OFFSETS}')

In [ ]:
# === TABEBM_CONFIGS 자동 조립 (모든 조합) ===
import itertools

TABEBM_CONFIGS = []
for steps, ss, ns, sn, dn, so in itertools.product(
    SGLD_STEPS, SGLD_STEP_SIZE, SGLD_NOISE_STD, STARTING_NOISE, DIST_NEG, SEED_OFFSETS):

    parts = []
    if len(SGLD_STEPS) > 1: parts.append(f'T{steps}')
    if len(SGLD_STEP_SIZE) > 1: parts.append(f'ss{ss}')
    if len(SGLD_NOISE_STD) > 1: parts.append(f'ns{ns}')
    if len(STARTING_NOISE) > 1: parts.append(f'sn{sn}')
    if len(DIST_NEG) > 1: parts.append(f'dn{dn}')
    if len(SEED_OFFSETS) > 1: parts.append(f's{so}')
    name = 'tabebm_' + ('_'.join(parts) if parts else 'default')

    TABEBM_CONFIGS.append(dict(
        name=name,
        sgld_steps=steps,
        sgld_step_size=ss,
        sgld_noise_std=ns,
        starting_point_noise_std=sn,
        distance_negative_class=dn,
        seed_offset=so,
    ))

print(f'{len(TABEBM_CONFIGS)} TabEBM configs:')
for c in TABEBM_CONFIGS:
    print(f'  {c["name"]:<35} steps={c["sgld_steps"]:<4d} '
          f'ss={c["sgld_step_size"]:<6g} ns={c["sgld_noise_std"]:<6g} '
          f'sn={c["starting_point_noise_std"]:<6g} dn={c["distance_negative_class"]:<5g} '
          f'so={c["seed_offset"]}')

## 3. 병렬화 + 생성 설정

In [ ]:
GPUS            = [0, 1, 2, 3]
# GPUS          = [0, 1]
# GPUS          = [0]

PROCS_PER_GPU   = 7
# PROCS_PER_GPU = 2

# Storage: 500 per class (abundance). 04 evaluator slices at eval time:
# 2-class biodeg → 총 1000 synthetic. 다른 class 수도 per-class 500.
N_SYN_PER_CLASS = 500
# N_SYN_PER_CLASS = 250
# N_SYN_PER_CLASS = 100

N_GPU = len(GPUS)
MAX_WORKERS = N_GPU * PROCS_PER_GPU
print(f'GPUs={GPUS}, workers={MAX_WORKERS}, N_syn_per_class={N_SYN_PER_CLASS}')


## 4. 샘플 생성 (병렬)

In [ ]:
from fair_eval_worker import run_one_sgld_task
from ensemble_ebm import apply_preprocessor, split_preprocessor_from_npz

data = np.load(EVAL_DIR / 'data.npz')
X_all_raw, y_all = data['X_all'], data['y_all']
sp = np.load(EVAL_DIR / 'splits.npz')
splits = [(sp[f'tr_{i}'], sp[f'val_{i}']) for i in range(config['n_splits'])]
classes = config['classes']

# Paper B.3: per-split preprocessor 적용
X_all_per_split = [
    apply_preprocessor(X_all_raw, split_preprocessor_from_npz(sp, i))
    for i in range(config['n_splits'])
]

ens_dir = str(EVAL_DIR / 'ensembles')
samples_dir = EVAL_DIR / 'samples'
samples_dir.mkdir(exist_ok=True)

# tabebm_sample_config.json merge (기존 유지 + 신규 추가)
_tc_path = EVAL_DIR / 'tabebm_sample_config.json'
if _tc_path.exists():
    _existing = json.loads(_tc_path.read_text())
    _existing_names = {c['name'] for c in _existing.get('tabebm_configs', [])}
else:
    _existing = {'tabebm_configs': []}
    _existing_names = set()
_new_configs = [c for c in TABEBM_CONFIGS if c['name'] not in _existing_names]
_merged = _existing.get('tabebm_configs', []) + _new_configs
_tc_path.write_text(json.dumps({
    'tabebm_configs': _merged,
    'n_syn_per_class': N_SYN_PER_CLASS,
}, indent=2))
print(f'tabebm_sample_config.json: 기존 {len(_existing_names)}개 + 신규 {len(_new_configs)}개 = {len(_merged)}개')

# task 구성 — 이미 존재하는 샘플은 skip
tasks = []; skipped = 0; task_id = 0
for split_i, (tr, _val) in enumerate(splits):
    X_all_s = X_all_per_split[split_i]
    for cfg in TABEBM_CONFIGS:
        for c in classes:
            out_path = samples_dir / f'split_{split_i}' / cfg['name'] / f'c{c}.npy'
            if out_path.exists():
                skipped += 1; task_id += 1; continue
            gpu = GPUS[task_id % N_GPU]
            tasks.append(('single', split_i, c, cfg,
                          tr, X_all_s, y_all, N_SYN_PER_CLASS,
                          config['seed'], gpu, ens_dir))
            task_id += 1

n_total = len(tasks)
print(f'{n_total} tasks (skipped {skipped} existing), {MAX_WORKERS} workers')
if n_total == 0:
    print('모든 샘플 이미 존재 — 생성 건너뜀')
else:
    print()
    t0 = time.time(); done = 0
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(run_one_sgld_task, t): t for t in tasks}
        for f in as_completed(futs):
            r = f.result()
            out = samples_dir / f'split_{r["split_i"]}' / r['cfg_name']
            out.mkdir(parents=True, exist_ok=True)
            np.save(out / f'c{r["class_c"]}.npy', r['samples'])
            done += 1
            if done % 10 == 0 or done == n_total:
                elapsed = time.time() - t0
                eta = elapsed / done * (n_total - done)
                print(f'  [{done:>3d}/{n_total}]  '
                      f'elapsed {_fmt(elapsed)}  ETA {_fmt(eta)}', flush=True)
    print(f'\nDone -- {_fmt(time.time()-t0)}')


## 5. 검증

In [31]:
expected = [c['name'] for c in TABEBM_CONFIGS]
ok = 0; fail = 0
for i in range(config['n_splits']):
    missing = []
    for cn in expected:
        for c in classes:
            if (samples_dir / f'split_{i}' / cn / f'c{c}.npy').exists():
                ok += 1
            else:
                missing.append(f'{cn}/c{c}'); fail += 1
    status = 'OK' if not missing else f'MISSING {missing[:3]}'
    print(f'  split_{i}: {status}')

print(f'\nTotal: {ok} OK, {fail} missing')
print(f'EVAL_DIR = {EVAL_DIR}  --> visualize / 04_evaluate 에 입력')

  split_0: OK
  split_1: OK
  split_2: OK
  split_3: OK
  split_4: OK
  split_5: OK
  split_6: OK
  split_7: OK
  split_8: OK
  split_9: OK

Total: 40 OK, 0 missing
EVAL_DIR = experiments/fair_eval/20260417_215937  --> visualize / 04_evaluate 에 입력
